# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ShamKottish/FlyRankML/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
%pip -q install duckdb huggingface_hub scikit-learn

import os
import getpass
import duckdb
import pandas as pd
import numpy as np

from sklearn.model_selection import (
    train_test_split,
    GroupShuffleSplit
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score
)

RANDOM_STATE = 42

# -----------------------------
# Hugging Face authentication
# -----------------------------

HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

HF_TOKEN = HF_TOKEN or getpass.getpass(
    "Paste your Hugging Face READ token: "
)

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

FACT_MAR = (
    f"read_parquet('{REL}/"
    "fact_content_daily_performance/month=2026-03/*.parquet')"
)

FACT_APR = (
    f"read_parquet('{REL}/"
    "fact_content_daily_performance/month=2026-04/*.parquet')"
)

print("✓ Connected")
print("Feature window: March 2026")
print("Outcome window: April 2026")
print("Random seed:", RANDOM_STATE)

Paste your Hugging Face READ token: ··········
✓ Connected
Feature window: March 2026
Outcome window: April 2026
Random seed: 42


In [2]:
# ============================================================
# REBUILD THE SAME MODELING FRAME USED IN ML-08
# ============================================================

march = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) AS feature_impressions,
        SUM(gsc_clicks) AS feature_clicks,

        CASE
            WHEN SUM(gsc_impressions) > 0
            THEN 100.0 * SUM(gsc_clicks)
                 / SUM(gsc_impressions)
        END AS feature_ctr,

        SUM(
            CASE
                WHEN gsc_avg_position > 0
                     AND gsc_impressions > 0
                THEN gsc_avg_position * gsc_impressions
                ELSE 0
            END
        )
        /
        NULLIF(
            SUM(
                CASE
                    WHEN gsc_avg_position > 0
                         AND gsc_impressions > 0
                    THEN gsc_impressions
                    ELSE 0
                END
            ),
            0
        ) AS feature_avg_position,

        STDDEV_SAMP(
            CASE
                WHEN gsc_avg_position > 0
                     AND gsc_impressions > 0
                THEN gsc_avg_position
            END
        ) AS feature_position_std,

        COUNT(
            DISTINCT CASE
                WHEN gsc_impressions > 0
                THEN report_date
            END
        ) AS feature_active_days

    FROM {FACT_MAR}

    GROUP BY
        client_hash_id,
        content_hash_id

    HAVING SUM(gsc_impressions) >= 500
""").df()

march["feature_position_std"] = (
    march["feature_position_std"].fillna(0)
)

march = march[
    march["feature_avg_position"].notna()
].copy()


# -----------------------------
# April outcome
# -----------------------------

april = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) AS outcome_impressions,
        SUM(gsc_clicks) AS outcome_clicks,

        CASE
            WHEN SUM(gsc_impressions) > 0
            THEN 100.0 * SUM(gsc_clicks)
                 / SUM(gsc_impressions)
        END AS outcome_ctr,

        SUM(
            CASE
                WHEN gsc_avg_position > 0
                     AND gsc_impressions > 0
                THEN gsc_avg_position * gsc_impressions
                ELSE 0
            END
        )
        /
        NULLIF(
            SUM(
                CASE
                    WHEN gsc_avg_position > 0
                         AND gsc_impressions > 0
                    THEN gsc_impressions
                    ELSE 0
                END
            ),
            0
        ) AS outcome_avg_position

    FROM {FACT_APR}

    GROUP BY
        client_hash_id,
        content_hash_id

    HAVING SUM(gsc_impressions) >= 500
""").df()

april = april[
    april["outcome_avg_position"].notna()
].copy()


def position_band(position):
    if position <= 3:
        return "top_3"
    elif position <= 10:
        return "page_1"
    elif position <= 20:
        return "striking"
    elif position <= 50:
        return "page_3_5"
    return "deep"


april["outcome_position_band"] = (
    april["outcome_avg_position"]
    .apply(position_band)
)

april["outcome_band_median_ctr"] = (
    april
    .groupby("outcome_position_band")["outcome_ctr"]
    .transform("median")
)

april["outcome_ctr_gap_pp"] = (
    april["outcome_band_median_ctr"]
    - april["outcome_ctr"]
)

april["opportunity_proxy"] = (
    april["outcome_ctr_gap_pp"] > 0.10
).astype(int)


# -----------------------------
# March -> April modeling frame
# -----------------------------

model_df = march.merge(
    april[
        [
            "client_hash_id",
            "content_hash_id",
            "outcome_impressions",
            "outcome_ctr",
            "outcome_avg_position",
            "outcome_ctr_gap_pp",
            "opportunity_proxy",
        ]
    ],
    on=[
        "client_hash_id",
        "content_hash_id"
    ],
    how="inner"
)

model_df["log_impressions"] = np.log1p(
    model_df["feature_impressions"]
)

model_df["log_clicks"] = np.log1p(
    model_df["feature_clicks"]
)

FEATURES = [
    "log_impressions",
    "log_clicks",
    "feature_ctr",
    "feature_avg_position",
    "feature_position_std",
    "feature_active_days",
]

TARGET = "opportunity_proxy"

print(f"Modeling rows: {len(model_df):,}")
print(
    f"Clients: "
    f"{model_df['client_hash_id'].nunique():,}"
)
print(
    f"Opportunity-proxy rate: "
    f"{model_df[TARGET].mean():.1%}"
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Modeling rows: 51,496
Clients: 34
Opportunity-proxy rate: 23.9%


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*


### Finding 1 — Which Pages Will Grow?

FlyRank reports a model trained on pages classified as clearly growing or declining. The underlying growth direction comes from observed search-performance movement rather than a human quality label. In the report, trend direction compares the most recent 30 days with the preceding 30 days.

The validation result is encouraging because performance is reported both for unseen pages from represented brands and for completely unseen brands. Performance is lower on unseen brands, which is important evidence that client-specific patterns affect generalization.

**My methodology question:** Were every model feature and the growth-direction label separated in time? If any feature summarizes the same 30-day period used to define the growth label, then the result is better described as identifying the current trajectory than forecasting future growth. A strict earlier-feature → later-outcome design would carry a stronger forecasting claim.

### Finding 2 — What Will Improve Next Month?

FlyRank also reports a 30-day momentum model whose outcome is whether a page improves by more than 10% in the following month. This is a stronger target design because the label refers to a later observed outcome rather than a current product decision.

The paper reports strong performance on both known-brand and completely unseen-brand tests, so the validation design provides useful evidence that the signal can generalize beyond pages seen during training.

**My methodology question:** In addition to holding out brands, was the final evaluation also chronological — training entirely on earlier periods and testing on a later untouched month? Because the claim is explicitly about "next month," a sealed future-time test would most closely reproduce deployment.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
paper_audit = pd.DataFrame([
    {
        "finding": "Which Pages Will Grow?",
        "label_source": (
            "Observed growth direction from search-performance movement"
        ),
        "strong_validation_feature": (
            "Reports performance on completely unseen brands"
        ),
        "methodology_question": (
            "Are all features strictly earlier than the "
            "growth-label window?"
        ),
    },
    {
        "finding": "What Will Improve Next Month?",
        "label_source": (
            "Observed later outcome: >10% improvement next month"
        ),
        "strong_validation_feature": (
            "Reports known-brand and unseen-brand tests"
        ),
        "methodology_question": (
            "Was the final evaluation also a chronological "
            "future-time holdout?"
        ),
    },
])

display(paper_audit)

assert len(paper_audit) == 2

print("✓ Two paper findings audited.")

,finding,label_source,strong_validation_feature,methodology_question
0,Which Pages Will Grow?,Observed growth direction from search-performa...,Reports performance on completely unseen brands,Are all features strictly earlier than the gro...
1,What Will Improve Next Month?,Observed later outcome: >10% improvement next ...,Reports known-brand and unseen-brand tests,Was the final evaluation also a chronological ...


✓ Two paper findings audited.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*


**Before — random page split:** individual content items are randomly divided between training and testing. This is convenient but potentially optimistic because pages belonging to the same client can appear in both sets.

**After — grouped client split:** entire clients are held out. A client appearing in training cannot appear in testing. This asks the harder and more useful question: can the model rank CTR opportunities for a client it has never seen?

Both versions use the same March features, the same April opportunity proxy, the same Random Forest settings, and Precision@50 as the main ranking metric.

The difference between the two results is itself useful evidence. If grouped performance is lower, that suggests some of the apparent skill from the random split came from client-specific patterns rather than fully transferable signal.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# ============================================================
# Helper functions
# ============================================================

def precision_at_k(y_true, scores, k=50):
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)

    k = min(k, len(y_true))

    order = np.argsort(-scores)

    return y_true[order[:k]].mean()


def evaluate(y_true, scores):
    return {
        "base_rate": y_true.mean(),
        "precision@20": precision_at_k(
            y_true, scores, 20
        ),
        "precision@50": precision_at_k(
            y_true, scores, 50
        ),
        "average_precision": average_precision_score(
            y_true, scores
        ),
        "roc_auc": roc_auc_score(
            y_true, scores
        ),
    }


def build_model():
    return RandomForestClassifier(
        n_estimators=300,
        max_depth=8,
        min_samples_leaf=10,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )


X = model_df[FEATURES]
y = model_df[TARGET]


# ============================================================
# BEFORE: RANDOM PAGE SPLIT
# ============================================================

random_train_idx, random_test_idx = train_test_split(
    np.arange(len(model_df)),
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=y,
)

X_train_random = X.iloc[random_train_idx]
X_test_random = X.iloc[random_test_idx]

y_train_random = y.iloc[random_train_idx]
y_test_random = y.iloc[random_test_idx]

random_model = build_model()

random_model.fit(
    X_train_random,
    y_train_random
)

random_scores = random_model.predict_proba(
    X_test_random
)[:, 1]

random_metrics = evaluate(
    y_test_random,
    random_scores
)


# How many clients leaked across the random split?
random_train_clients = set(
    model_df.iloc[random_train_idx]["client_hash_id"]
)

random_test_clients = set(
    model_df.iloc[random_test_idx]["client_hash_id"]
)

random_client_overlap = (
    random_train_clients
    .intersection(random_test_clients)
)


# ============================================================
# AFTER: GROUPED CLIENT HOLDOUT
# ============================================================

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=RANDOM_STATE,
)

group_train_idx, group_test_idx = next(
    gss.split(
        X,
        y,
        groups=model_df["client_hash_id"]
    )
)

X_train_group = X.iloc[group_train_idx]
X_test_group = X.iloc[group_test_idx]

y_train_group = y.iloc[group_train_idx]
y_test_group = y.iloc[group_test_idx]

group_model = build_model()

group_model.fit(
    X_train_group,
    y_train_group
)

group_scores = group_model.predict_proba(
    X_test_group
)[:, 1]

group_metrics = evaluate(
    y_test_group,
    group_scores
)


group_train_clients = set(
    model_df.iloc[group_train_idx]["client_hash_id"]
)

group_test_clients = set(
    model_df.iloc[group_test_idx]["client_hash_id"]
)

group_client_overlap = (
    group_train_clients
    .intersection(group_test_clients)
)

assert len(group_client_overlap) == 0


# ============================================================
# COMPARISON
# ============================================================

comparison = pd.DataFrame([
    {
        "validation": "random_page_split",
        **random_metrics,
        "client_overlap": len(random_client_overlap),
    },
    {
        "validation": "grouped_client_holdout",
        **group_metrics,
        "client_overlap": len(group_client_overlap),
    },
])

metric_cols = [
    "base_rate",
    "precision@20",
    "precision@50",
    "average_precision",
    "roc_auc",
]

comparison[metric_cols] = (
    comparison[metric_cols].round(3)
)

display(comparison)

print(
    "Clients appearing in BOTH sides of random split:",
    len(random_client_overlap)
)

print(
    "Clients appearing in BOTH sides of grouped split:",
    len(group_client_overlap)
)

random_p50 = random_metrics["precision@50"]
grouped_p50 = group_metrics["precision@50"]

print(
    f"\nRandom-split Precision@50: "
    f"{random_p50:.3f}"
)

print(
    f"Grouped Precision@50:      "
    f"{grouped_p50:.3f}"
)

print(
    f"Change after honest split: "
    f"{grouped_p50 - random_p50:+.3f}"
)

,validation,base_rate,precision@20,precision@50,average_precision,roc_auc,client_overlap
0,random_page_split,0.239,0.85,0.92,0.638,0.860,30
1,grouped_client_holdout,0.215,0.85,0.88,0.611,0.865,0


Clients appearing in BOTH sides of random split: 30
Clients appearing in BOTH sides of grouped split: 0

Random-split Precision@50: 0.920
Grouped Precision@50:      0.880
Change after honest split: -0.040


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*


I repeat the leakage audit on the final model rather than assuming the Week-3 feature list is still safe.

The final feature window is March 1–31, 2026. The April CTR-opportunity proxy is measured strictly afterward.

I check four risks:

1. **Future-window leakage:** no April or outcome fields may appear in the feature set.
2. **Label-derived leakage:** `outcome_ctr`, `outcome_ctr_gap_pp`, and `opportunity_proxy` are forbidden because they directly contain or define the answer.
3. **Identifier leakage:** pseudonymized client and content IDs are used for grouping and joins only, not as predictive inputs.
4. **Decision/privacy leakage:** no FlyRank product scores or flags, raw client names, domains, URLs, titles, or private queries are model features.

As a stress test, I deliberately add `outcome_ctr_gap_pp`, which directly contributes to the April label, to a temporary model. If the validation score becomes unrealistically strong, that confirms the leakage detector is working. The deliberately leaky model is diagnostic only and is never used as a reported model.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ============================================================
# STATIC LEAKAGE AUDIT
# ============================================================

feature_set = set(FEATURES)

future_or_label_terms = [
    "outcome",
    "april",
    "proxy",
    "gap_pp",
]

future_leaks = sorted([
    col
    for col in feature_set
    if any(
        term in col.lower()
        for term in future_or_label_terms
    )
])

identifier_leaks = sorted(
    feature_set.intersection({
        "client_hash_id",
        "content_hash_id",
        "url_hash_id",
        "keyword_hash_id",
    })
)

product_terms = [
    "health_score",
    "priority_score",
    "action_type",
    "needs_ctr_fix",
    "is_quick_win",
    "refresh_flag",
]

product_leaks = sorted([
    col
    for col in feature_set
    if any(
        term in col.lower()
        for term in product_terms
    )
])

private_terms = [
    "client_name",
    "domain",
    "raw_url",
    "raw_query",
    "content_title",
]

privacy_leaks = sorted([
    col
    for col in feature_set
    if any(
        term in col.lower()
        for term in private_terms
    )
])

print("Future / label-derived leaks:", future_leaks)
print("Identifier leaks:", identifier_leaks)
print("Product decision leaks:", product_leaks)
print("Privacy leaks:", privacy_leaks)

assert future_leaks == []
assert identifier_leaks == []
assert product_leaks == []
assert privacy_leaks == []

FEATURE_END = pd.Timestamp("2026-03-31")
OUTCOME_START = pd.Timestamp("2026-04-01")

assert FEATURE_END < OUTCOME_START

print("\n✓ Timeline is ordered correctly.")
print("✓ Final feature set passes static leakage checks.")


# ============================================================
# DELIBERATE LEAK STRESS TEST
# ============================================================

# Use the exact grouped split from Section 2.
leaky_features = FEATURES + [
    "outcome_ctr_gap_pp"
]

X_leaky_train = (
    model_df.iloc[group_train_idx][leaky_features]
)

X_leaky_test = (
    model_df.iloc[group_test_idx][leaky_features]
)

leaky_model = build_model()

leaky_model.fit(
    X_leaky_train,
    y_train_group
)

leaky_scores = leaky_model.predict_proba(
    X_leaky_test
)[:, 1]

leaky_metrics = evaluate(
    y_test_group,
    leaky_scores
)

leak_test = pd.DataFrame([
    {
        "model": "honest_features",
        "precision@50": grouped_p50,
        "average_precision":
            group_metrics["average_precision"],
    },
    {
        "model": "DELIBERATELY_LEAKY",
        "precision@50":
            leaky_metrics["precision@50"],
        "average_precision":
            leaky_metrics["average_precision"],
    }
]).round(3)

display(leak_test)

print(
    "\nThe deliberately leaky model is NOT "
    "a valid result and must not be reported "
    "as model performance."
)

Future / label-derived leaks: []
Identifier leaks: []
Product decision leaks: []
Privacy leaks: []

✓ Timeline is ordered correctly.
✓ Final feature set passes static leakage checks.


,model,precision@50,average_precision
0,honest_features,0.88,0.611
1,DELIBERATELY_LEAKY,1.00,1.000



The deliberately leaky model is NOT a valid result and must not be reported as model performance.


## 4. Claim rewrite

### Too strong

> My model predicts which pages need CTR optimization and which edits will improve their performance.

This claim is too strong because the model only ranks pages associated with a later observed CTR-opportunity proxy. It does not observe the causal effect of editing a page and therefore cannot show that a specific change will improve CTR, traffic, or rankings.

### Safe research claim

Using March 2026 observed search-performance signals, my model ranked pages associated with a position-adjusted CTR-opportunity proxy measured in April 2026.

On completely held-out clients, the model achieved a **Precision@50 of 0.880**, compared with a test-population opportunity base rate of **0.215 (21.5%)**. This means that **44 of the first 50 pages** ranked by the model matched the later April opportunity proxy in this held-out evaluation.

The model also achieved an **Average Precision of 0.611** using the final honest feature set. These results provide evidence that the March search-performance signals contain useful information for prioritizing pages for human review.

The findings should be interpreted as **observational, directional, and decision-support evidence**. They do not establish that a recommended page is objectively poor, that it must be edited, or that changing its title, metadata, content, or other SEO elements will cause CTR or traffic to improve.

Performance may also differ across future time periods, new client populations, changing SERP conditions, seasonality, and other factors not directly observed by the model.

For these reasons, the model should be used to help an SEO specialist or content editor decide **which pages to inspect first**, while the final decision to make any change remains with a human reviewer.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
grouped_p50 = group_metrics["precision@50"]
grouped_base_rate = group_metrics["base_rate"]

safe_claim = (
    "Using March 2026 observed search-performance signals, "
    "the model ranked pages associated with a "
    "position-adjusted CTR-opportunity proxy measured in "
    "April 2026. On completely held-out clients, "
    f"Precision@50 was {grouped_p50:.3f}, compared with "
    f"a test-population base rate of "
    f"{grouped_base_rate:.3f}. "
    "This is observational, directional decision-support; "
    "it does not show that editing a recommended page "
    "causes CTR or traffic to improve."
)

print("SAFE CLAIM:\n")
print(safe_claim)

SAFE CLAIM:

Using March 2026 observed search-performance signals, the model ranked pages associated with a position-adjusted CTR-opportunity proxy measured in April 2026. On completely held-out clients, Precision@50 was 0.880, compared with a test-population base rate of 0.215. This is observational, directional decision-support; it does not show that editing a recommended page causes CTR or traffic to improve.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.